# Regression Track — Notebook 01
# 1. Introduction  ·  2. Import Libraries  ·  3. Load Dataset

**Course:** 23CSE301 Machine Learning — Capstone Project (Review 1)
**Track:** Regression — Seoul Bike Sharing Demand

---

### How this project is organised

The capstone is split into **15 focused notebooks** instead of one very long
file. Each notebook owns one stage of the pipeline, runs top-to-bottom on its
own, and hands its output to the next notebook through the `artifacts/`
folder. Notebooks 01–09 are the Regression track, 10–14 are Classification
Part A, and notebook 15 holds the viva preparation and the final checklist.

> **Run order matters.** Each notebook loads the artifact written by the
> previous one. Run them in numerical order the first time.

---
## 1. Project Introduction

### 1.1 Project title

**Predicting Hourly Public Bicycle Rental Demand in Seoul using Supervised
Regression Models**

### 1.2 Problem statement

Seoul operates a public bike-sharing system in which bicycles are distributed
across rental stations throughout the city. The number of bikes people want to
rent changes dramatically from hour to hour and from season to season. If too
few bikes are available at a given hour, customers are turned away and revenue
is lost; if too many are deployed, the operator pays for redistribution
trucks, staff, and maintenance on bicycles that nobody uses.

The operational question is therefore: **given the calendar information and
the weather conditions for a particular hour, how many bicycles will be
rented in that hour?**

### 1.3 Business / real-world relevance

| Stakeholder | Why an hourly demand forecast matters |
|---|---|
| City transport authority | Plan bike supply so that "stable supply of rental bikes" is actually achieved |
| Operations team | Schedule redistribution trucks before predicted peak hours rather than after |
| Maintenance team | Service bicycles during predicted low-demand windows |
| Finance / planning | Estimate revenue and justify fleet expansion with evidence |

### 1.4 Objective

1. Build a clean, leakage-free preprocessing pipeline for the Seoul bike data.
2. Train **ten** different regression algorithms on an identical train/test split.
3. Compare them fairly on **R², RMSE and MAE**.
4. Tune at least two models with cross-validated search on the training data only.
5. Analyse the final model's residuals and feature importances, and state its limitations honestly.

### 1.5 Target variable

**`Rented Bike Count`** — the number of bikes rented during a given hour.
It is a non-negative **count / continuous numeric** quantity.

### 1.6 Why regression is appropriate

The target is a numeric quantity on a continuous-like scale (0 to ~3,556),
not a label from a fixed set of categories. The cost of an error grows with
how far the prediction is from the truth — predicting 900 when the truth is
890 is nearly harmless, predicting 100 is not. That is precisely the setting
regression is designed for, and it is why we score with R², RMSE and MAE
rather than with accuracy.

### 1.7 Dataset source

* **Name:** Seoul Bike Sharing Demand Data Set
* **Repository:** UCI Machine Learning Repository
* **URL:** <https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand>
* **File used:** `dataset/SeoulBikeData.csv`
* **Coverage:** 1 December 2017 – 30 November 2018, one row per hour

### 1.8 Dataset description

| Column | Type | Meaning |
|---|---|---|
| `Date` | text (DD/MM/YYYY) | Calendar date of the observation |
| `Rented Bike Count` | integer | **TARGET** — bikes rented in that hour |
| `Hour` | integer 0–23 | Hour of the day |
| `Temperature(°C)` | float | Air temperature |
| `Humidity(%)` | integer | Relative humidity |
| `Wind speed (m/s)` | float | Wind speed |
| `Visibility (10m)` | integer | Visibility in units of 10 m |
| `Dew point temperature(°C)` | float | Dew point temperature |
| `Solar Radiation (MJ/m2)` | float | Solar radiation |
| `Rainfall(mm)` | float | Rainfall in that hour |
| `Snowfall (cm)` | float | Snowfall in that hour |
| `Seasons` | category | Winter / Spring / Summer / Autumn |
| `Holiday` | category | Holiday / No Holiday |
| `Functioning Day` | category | Whether the rental system was operating |

---
## 2. Import Libraries

In [1]:
# --- Standard library -------------------------------------------------------
import sys
import warnings
from pathlib import Path

# --- Third-party scientific stack ------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# --- Make the shared helper module importable from any working directory ----
def find_src_folder() -> Path:
    """Walk upwards from the current folder until src/ml_utils.py is found."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "ml_utils.py").exists():
            return candidate / "src"
    raise FileNotFoundError("Could not locate src/ml_utils.py — check the repo layout.")


sys.path.insert(0, str(find_src_folder()))

# --- Project helpers (shared by all 15 notebooks) ---------------------------
from ml_utils import (  # noqa: E402
    DATA_DIR,
    RANDOM_STATE,
    ROOT,
    audit_table,
    save_artifact,
    section,
    set_plot_style,
)

warnings.filterwarnings("ignore", category=FutureWarning)
set_plot_style()

print(f"Project root : {ROOT}")
print(f"Data folder  : {DATA_DIR}")
print(f"Random state : {RANDOM_STATE}")
print()
print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"matplotlib  {plt.matplotlib.__version__}")
print(f"seaborn     {sns.__version__}")

import sklearn  # noqa: E402

print(f"scikit-learn {sklearn.__version__}")

Project root : /home/claude/project
Data folder  : /home/claude/project/data
Random state : 42

pandas      3.0.2
numpy       2.4.4
matplotlib  3.10.8
seaborn     0.13.2
scikit-learn 1.8.0


---
## 3. Load Dataset

### 3.1 Reading the CSV

The file is **not** UTF-8. Its column headers contain the degree symbol
(`Temperature(°C)`), which was written with the Latin-1 / CP-1252 code page.
Reading it with the pandas default raises a `UnicodeDecodeError`, so we state
the encoding explicitly. This is a real data-engineering detail, not a
cosmetic one — silently guessing the encoding is how column names get
corrupted.

In [2]:
CSV_PATH = DATA_DIR / "SeoulBikeData.csv"

bike_df = pd.read_csv(CSV_PATH, encoding="cp1252")

print(f"Loaded: {CSV_PATH.name}")
print(f"Rows: {bike_df.shape[0]:,}   Columns: {bike_df.shape[1]}")

Loaded: SeoulBikeData.csv
Rows: 8,760   Columns: 14


### 3.2 First five rows

In [3]:
bike_df.head()

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


### 3.3 Last five rows

In [4]:
bike_df.tail()

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
8755,30/11/2018,1003,19,4.2,34,2.6,1894,-10.3,0.0,0.0,0.0,Autumn,No Holiday,Yes
8756,30/11/2018,764,20,3.4,37,2.3,2000,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8757,30/11/2018,694,21,2.6,39,0.3,1968,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8758,30/11/2018,712,22,2.1,41,1.0,1859,-9.8,0.0,0.0,0.0,Autumn,No Holiday,Yes
8759,30/11/2018,584,23,1.9,43,1.3,1909,-9.3,0.0,0.0,0.0,Autumn,No Holiday,Yes


**Observation.** The first row is 01/12/2017 hour 0 and the last row is
30/11/2018 hour 23. The data is an unbroken hourly time series covering one
full year: 365 days × 24 hours = 8,760 rows, which matches the shape exactly.

### 3.4 Shape, column names and data types

In [5]:
section("SHAPE")
print(f"Rows    : {bike_df.shape[0]:,}")
print(f"Columns : {bike_df.shape[1]}")

section("COLUMN NAMES")
for i, col in enumerate(bike_df.columns, start=1):
    print(f"{i:>2}. {col}")

section("DATA TYPES")
print(bike_df.dtypes.to_string())

SHAPE
Rows    : 8,760
Columns : 14
COLUMN NAMES
 1. Date
 2. Rented Bike Count
 3. Hour
 4. Temperature(°C)
 5. Humidity(%)
 6. Wind speed (m/s)
 7. Visibility (10m)
 8. Dew point temperature(°C)
 9. Solar Radiation (MJ/m2)
10. Rainfall(mm)
11. Snowfall (cm)
12. Seasons
13. Holiday
14. Functioning Day
DATA TYPES
Date                             str
Rented Bike Count              int64
Hour                           int64
Temperature(°C)              float64
Humidity(%)                    int64
Wind speed (m/s)             float64
Visibility (10m)               int64
Dew point temperature(°C)    float64
Solar Radiation (MJ/m2)      float64
Rainfall(mm)                 float64
Snowfall (cm)                float64
Seasons                          str
Holiday                          str
Functioning Day                  str


### 3.5 `info()`

In [6]:
bike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date                       8760 non-null   str    
 1   Rented Bike Count          8760 non-null   int64  
 2   Hour                       8760 non-null   int64  
 3   Temperature(°C)            8760 non-null   float64
 4   Humidity(%)                8760 non-null   int64  
 5   Wind speed (m/s)           8760 non-null   float64
 6   Visibility (10m)           8760 non-null   int64  
 7   Dew point temperature(°C)  8760 non-null   float64
 8   Solar Radiation (MJ/m2)    8760 non-null   float64
 9   Rainfall(mm)               8760 non-null   float64
 10  Snowfall (cm)              8760 non-null   float64
 11  Seasons                    8760 non-null   str    
 12  Holiday                    8760 non-null   str    
 13  Functioning Day            8760 non-null   str    
dtypes: 

**Observation.** All 8,760 entries are non-null in every column, so there are
no *encoded-as-NaN* missing values. `Date`, `Seasons`, `Holiday` and
`Functioning Day` arrive as text; everything else is already numeric.

**Interpretation.** `Date` being text is a problem we must fix — as a string
it is useless to a model, but the calendar information inside it (month, day
of week, weekend) is likely to be highly predictive. That conversion is done
in Notebook 05 (Feature Engineering). The three text columns are genuine
categorical variables and will need encoding before modelling.

### 3.6 Descriptive statistics — numerical columns

In [7]:
bike_df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Rented Bike Count,8760.0,704.60,645.00,0.0,191.00,504.50,1065.25,3556.00
Hour,8760.0,11.50,6.92,0.0,5.75,11.50,17.25,23.00
Temperature(°C),8760.0,12.88,11.94,-17.8,3.50,13.70,22.50,39.40
Humidity(%),8760.0,58.23,20.36,0.0,42.00,57.00,74.00,98.00
Wind speed (m/s),8760.0,1.72,1.04,0.0,0.90,1.50,2.30,7.40
Visibility (10m),8760.0,1436.83,608.30,27.0,940.00,1698.00,2000.00,2000.00
Dew point temperature(°C),8760.0,4.07,13.06,-30.6,-4.70,5.10,14.80,27.20
Solar Radiation (MJ/m2),8760.0,0.57,0.87,0.0,0.00,0.01,0.93,3.52
Rainfall(mm),8760.0,0.15,1.13,0.0,0.00,0.00,0.00,35.00
Snowfall (cm),8760.0,0.08,0.44,0.0,0.00,0.00,0.00,8.80


**Observation.**

* `Rented Bike Count` ranges from **0 to 3,556** with a mean near 705 and a
  median near 505. Mean > median means the distribution is **right-skewed**.
* `Rainfall(mm)` and `Snowfall (cm)` are 0 at the 75th percentile — they are
  mostly zero with occasional spikes.
* `Solar Radiation` is 0 at the median, which is expected: it is zero at night.
* `Temperature(°C)` spans −17.8 °C to 39.4 °C, a full Korean year.

**Interpretation.** A right-skewed target with many near-zero hours and a
long upper tail suggests the relationship between the features and the target
will not be a straight line. We should expect flexible, non-linear models
(trees, ensembles) to do better than plain Linear Regression — a hypothesis
we will test rather than assume.

### 3.7 Descriptive statistics — categorical columns

In [8]:
bike_df.describe(include="object").T

/tmp/ipykernel_214/489828963.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  bike_df.describe(include="object").T


,count,unique,top,freq
Date,8760,365,01/12/2017,24
Seasons,8760,4,Spring,2208
Holiday,8760,2,No Holiday,8328
Functioning Day,8760,2,Yes,8465


### 3.8 Quick per-column audit

A compact version of the audit; Notebook 02 does this properly and in depth.

In [9]:
audit_table(bike_df)

,Column,Dtype,Missing,Missing %,Unique,Example value
0,Date,str,0,0.0,365,01/12/2017
1,Rented Bike Count,int64,0,0.0,2166,254
2,Hour,int64,0,0.0,24,0
3,Temperature(°C),float64,0,0.0,546,-5.2
4,Humidity(%),int64,0,0.0,90,37
5,Wind speed (m/s),float64,0,0.0,65,2.2
6,Visibility (10m),int64,0,0.0,1789,2000
7,Dew point temperature(°C),float64,0,0.0,556,-17.6
8,Solar Radiation (MJ/m2),float64,0,0.0,345,0.0
9,Rainfall(mm),float64,0,0.0,61,0.0


---
## Handover to Notebook 02

The raw DataFrame is saved **unmodified** so that the data-quality audit in
Notebook 02 inspects exactly what the CSV contained. No rows have been
dropped, no values changed, no columns added.

In [10]:
save_artifact(bike_df, "reg_01_raw")

print("\nNotebook 01 complete.")
print("Next: 02_regression_data_quality_audit.ipynb")

[saved]  artifacts/reg_01_raw.joblib

Notebook 01 complete.
Next: 02_regression_data_quality_audit.ipynb
